# 07 - Second submission: post-processing rule on the stage-2 ranking

Optional. Run notebook 06 first. This notebook leaves the stage-2 records untouched and writes a second file, `submission/submission_jitter.csv`.

**Rule.** If `NONE` is among the first two candidates of stage 1 and also among the first two codes of stage 2, and `ANS-01` is not among the first two candidates of stage 1, then `NONE` becomes the first code of the stage-2 ranking. The other codes keep their order. Every other row is unchanged.

The rule was found by cross-tabulating the outputs of the stages against each other. It uses no model and no GPU.

**Reads** `submission/submission.csv`, `work/stage1/records`, `work/stage2/records`.  **Writes** `submission/submission_jitter.csv`, `submission/jitter_changes.csv` (one line per changed row).

In [ ]:
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
os.chdir(REPO); sys.path.insert(0, str(REPO / 'src'))
from misc_pipeline import common, jitter
print('repo:', REPO, '| work:', common.WORK)

In [ ]:
path, changes = jitter.make()
print(len(changes), 'rows changed')
for c in changes[:10]: print(c)

In [ ]:
rows = common.read_csv(path)
sample = common.read_csv(common.DATA / 'sample_submission.csv')
assert [r['trace_id'] for r in rows] == [r['trace_id'] for r in sample] and len(rows) == 3498
valid = {r['id'] for r in common.read_csv(common.DATA / 'taxonomy_v1.csv')} | {'NONE'}
assert all(1 <= len(r['root_code'].split()) <= 3 and set(r['root_code'].split()) <= valid for r in rows)
print('submission format OK:', len(rows), 'rows ->', path)